[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sampriti-sr/MLP_Course/blob/kshitij-variant/docs/recsys/06-implicit-retail.ipynb)

# Implicit Feedback on Retail Purchases

<div class="admonition info">
<p class="admonition-title">What you will be able to do after this page</p>
<ul>
<li>Turn purchase lines into a customer-by-product matrix and choose how to weight repeated purchases.</li>
<li>Evaluate a recommender by date, with two different definitions of a hit, and say which business question each answers.</li>
<li>Check a collaborative model against the strongest simple baseline for the question it is meant to answer.</li>
<li>Explain why a choice that wins on one validation window can fail on the test window.</li>
</ul>
</div>

**Prerequisites:** [Matrix factorisation](05-factorisation.ipynb) and the [lifecycle chapter's cleaning page](../lifecycle/03-cleaning.ipynb). **Dataset:** Online Retail II (cleaned). **Time:** about 45 minutes.

## The situation

The retailer sells to shops and wholesalers, and sales staff want a **"customers like you also bought"** list for each account. There are no ratings here. A customer never says "I liked this"; they *buy*, and the data records what, when, and how often. This is **implicit feedback**: every observation is a positive signal of unknown strength, and *the absence of a purchase is not a dislike*, only "not yet, or not known".

In [1]:
try:
    import mlp_data
except ImportError:  # e.g. a fresh Colab runtime
    %pip install -q "mlp-course @ git+https://github.com/sampriti-sr/MLP_Course@kshitij-variant"
    import mlp_data

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlp_data import load_retail, retail, recsys as rs, style

style.apply()
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

## Maths ↔ code

The implicit-feedback ALS of [page 5](05-factorisation.ipynb) takes a matrix $x_{ui}$ of purchase counts and a confidence $c_{ui} = 1 + \alpha\,x_{ui}$ (`rs.als_implicit(counts, alpha=...)`). A purchase count of 20 says "this customer is serious about this product"; a count of 1 might be an accident. How strongly to believe the counts is the confidence weight $\alpha$, and whether to feed the model the raw count, a $\log(1 + x)$ of it, or just 0/1 is a modelling choice to be tuned.

Dates replace the per-user split of the MovieLens pages: **everything before a cutoff date is history, the next 90 days is the future** (`rs.make_purchase_task`). This is the same time-ordered idea, and it uses the same cutoffs as the lifecycle chapter (validation on 1 June 2011, test on 1 September 2011).

## 0. Setup

In [2]:
cleaned = retail.clean_retail(load_retail())
sales = cleaned.sales
index = rs.make_index(sales, "customer_id", "stock_code")

validation = rs.make_purchase_task(sales, retail.VALID_CUTOFF, days=90, index=index)
test = rs.make_purchase_task(sales, retail.TEST_CUTOFF, days=90, index=index)
n_customers, n_products = test.B.shape
print(f"{len(sales):,} purchase lines, {n_customers:,} customers, {n_products:,} products; matrix density {test.B.nnz / (n_customers * n_products):.2%} at the test cutoff")

def existing_only(series, t):
    """Keep customers with at least one purchase before the cutoff. A model that learns from history has nothing to say
    about the others (every score is zero), so they are served by a fallback and reported separately."""
    has_history = np.asarray(t.B.sum(axis=1)).ravel() > 0
    return series[has_history[series.index.to_numpy()]]

def window_summary(t):
    has_history = np.asarray(t.B.sum(axis=1)).ravel() > 0
    buys = np.asarray(t.repeat.sum(axis=1)).ravel() > 0
    buys_new = np.asarray(t.new.sum(axis=1)).ravel() > 0
    return {"window": f"{t.window[0]:%d %b} to {t.window[1]:%d %b %Y}", "customers with history": int(has_history.sum()),
            "customers who buy in the window": int(buys.sum()), "... of them with no history": int((buys & ~has_history).sum()),
            "... buying something new to them": int(buys_new.sum()),
            "share of bought pairs that are repeats": 1 - t.new.nnz / t.repeat.nnz}
pd.DataFrame({"validation": window_summary(validation), "test": window_summary(test)})

787,223 purchase lines, 5,850 customers, 4,617 products; matrix density 1.43% at the test cutoff


,validation,test
window,01 Jun to 30 Aug 2011,01 Sep to 30 Nov 2011
customers with history,4905,5221
customers who buy in the window,1925,2831
... of them with no history,309,598
... buying something new to them,1827,2740
share of bought pairs that are repeats,0.3753,0.2912


The test window starts on 1 September 2011: 5,221 customers have a history, 2,831 of them buy something in the next 90 days, and **598 of those buyers (21%) have no history at all**, so no collaborative model can say anything about them (section 7). Of the (customer, product) pairs bought in the window, **29% are repeats** (38% in the validation window). Repurchase is a large part of this business, which is why the two questions below matter separately. The matrix is 1.4% dense, close to MovieLens' 1.7%.

## 1. Two different questions

Because a customer can buy the same product again, "what will they buy next?" splits in two, and each needs its own definition of a hit:

| Question | Hit | What may be recommended | Business use |
|---|---|---|---|
| **Repurchase**: what will they order again? | any product bought in the window | everything, including products they bought before | a pre-filled reorder form |
| **Discovery**: what could we introduce? | a product bought in the window **that they had never bought** | only products new to them | cross-selling, "you may also like" |

A model can be good at one and useless at the other, so each is scored separately.

## 2. Weighting the purchases, and tuning on validation

Three ways to feed purchase history to the implicit ALS model: **0/1** ("ever bought"), **raw lines** (how many order lines), and $\log(1+\text{lines})$. Tune each on the validation window, scored on *discovery*.

In [3]:
def signals(t):
    logged = t.counts.copy(); logged.data = np.log1p(logged.data)
    return {"0/1 (ever bought)": t.B, "log(1 + lines)": logged, "raw lines": t.counts}

def discovery_per_customer(scores, t):
    return existing_only(rs.evaluate(scores, t.B, t.new)["ndcg"], t)

def discovery_ndcg(scores, t):
    return discovery_per_customer(scores, t).mean()

rows = []
for name, signal in signals(validation).items():
    for factors, reg, alpha in itertools.product((16, 64), (10, 50), (1, 10)):
        scores = rs.als_implicit(signal, factors=factors, reg=reg, alpha=alpha, iters=10, seed=0).scores()
        rows.append({"signal": name, "factors": factors, "reg": reg, "alpha": alpha, "validation NDCG@10": discovery_ndcg(scores, validation)})
tuning = pd.DataFrame(rows)
best = tuning.loc[tuning.groupby("signal")["validation NDCG@10"].idxmax()].set_index("signal")
best_signal = best["validation NDCG@10"].idxmax()
chosen = best.loc[best_signal]

knn_grid = {(k, lam): discovery_ndcg(rs.item_knn_scores(validation.B, rs.item_similarity(validation.B, k=k, shrink=lam)), validation)
            for k in (20, 50, 100) for lam in (0, 10, 50)}
best_k, best_lam = max(knn_grid, key=knn_grid.get)
print(f"item-item on validation: best k = {best_k}, shrinkage = {best_lam}, NDCG@10 = {knn_grid[(best_k, best_lam)]:.4f}")
print(f"validation popularity: {discovery_ndcg(rs.broadcast(rs.popularity(validation.B), n_customers), validation):.4f}")
best

item-item on validation: best k = 50, shrinkage = 50, NDCG@10 = 0.0872


validation popularity: 0.0429


,factors,reg,alpha,validation NDCG@10
signal,,,,
0/1 (ever bought),64,10,1,0.0781
log(1 + lines),64,10,1,0.0795
raw lines,64,50,1,0.0809


Unlike on MovieLens, **the weighting hardly matters**: on validation the three signals score 0.0781 (0/1), 0.0795 (log) and 0.0809 (raw lines), a spread of 0.003. We take the best, raw lines with 64 factors, regularisation 50 and a low confidence weight of 1 (64 factors is the edge of the grid; a larger grid might find a little more). For reference, on validation item-item scores 0.0872 and popularity 0.0429.

## 3. Discovery on the test window

In [4]:
def recent_popularity(t, days):
    recent = sales[(sales["invoice_date"] < t.window[0]) & (sales["invoice_date"] >= t.window[0] - pd.Timedelta(days=days))]
    m = rs.to_csr(recent, index, user="customer_id", item="stock_code")
    m.data[:] = 1.0
    return rs.popularity(m)

signal = signals(test)[best_signal]
S_item = rs.item_similarity(test.B, k=best_k, shrink=best_lam)
discovery = {
    "popularity (all history)": rs.broadcast(rs.popularity(test.B), n_customers),
    "popularity (last 30 days)": rs.broadcast(recent_popularity(test, 30), n_customers),
    "PureSVD, 20 factors": rs.pure_svd(test.B, 20),
    "item-item": rs.item_knn_scores(test.B, S_item),
    "implicit ALS": rs.als_implicit(signal, factors=int(chosen["factors"]), reg=chosen["reg"], alpha=chosen["alpha"], iters=10, seed=0).scores(),
    "random list": np.random.default_rng(0).random(test.B.shape),
}
per_customer = {name: discovery_per_customer(S, test) for name, S in discovery.items()}
reference = per_customer["popularity (all history)"]
rows = {}
for name, s in per_customer.items():
    d = rs.paired_difference(s, reference)
    rows[name] = {"test NDCG@10 (discovery)": s.mean(),
                  "vs popularity (all history)": "" if name == "popularity (all history)" else f"{d['diff']:+.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}]"}
discovery_table = pd.DataFrame(rows).T
for name in ("PureSVD, 20 factors", "item-item", "implicit ALS", "popularity (last 30 days)"):          # guards the reading below
    assert rs.paired_difference(per_customer[name], reference)["lo"] > 0
print(f"{len(reference):,} existing customers scored (the {int(((np.asarray(test.repeat.sum(axis=1)).ravel() > 0) & (np.asarray(test.B.sum(axis=1)).ravel() == 0)).sum())} new customers who buy are left to a fallback; see section 7)")
d_als_knn = rs.paired_difference(per_customer["implicit ALS"], per_customer["item-item"])
d_als_svd = rs.paired_difference(per_customer["implicit ALS"], per_customer["PureSVD, 20 factors"])
print(f"implicit ALS minus item-item: {d_als_knn['diff']:+.4f} [{d_als_knn['lo']:+.4f}, {d_als_knn['hi']:+.4f}]; "
      f"minus PureSVD: {d_als_svd['diff']:+.4f} [{d_als_svd['lo']:+.4f}, {d_als_svd['hi']:+.4f}]")
assert d_als_svd["lo"] > 0 and d_als_knn["lo"] < 0                                    # guards the reading below
discovery_table

2,142 existing customers scored (the 598 new customers who buy are left to a fallback; see section 7)
implicit ALS minus item-item: +0.0039 [-0.0001, +0.0080]; minus PureSVD: +0.0057 [+0.0025, +0.0088]


,test NDCG@10 (discovery),vs popularity (all history)
popularity (all history),0.0436,
popularity (last 30 days),0.0531,"+0.0095 [+0.0052, +0.0138]"
"PureSVD, 20 factors",0.0803,"+0.0367 [+0.0312, +0.0421]"
item-item,0.0820,"+0.0384 [+0.0326, +0.0442]"
implicit ALS,0.0860,"+0.0423 [+0.0369, +0.0478]"
random list,0.0063,"-0.0373 [-0.0411, -0.0335]"


Only **existing customers** are scored here (2,142 of them), because a model that learns from history has nothing to say about the 598 newcomers who buy; section 7 covers them.

All three collaborative models are **about twice as good as popularity**: implicit ALS scores 0.0860 against 0.0436 (+0.0423, interval [+0.0369, +0.0478]), item-item 0.0820 (+0.0384) and PureSVD 0.0803 (+0.0367). The recent-popularity baseline helps a little (+0.0095, interval [+0.0052, +0.0138]); the random list is far below. Among the collaborative models the ranking is not sharp: ALS is **reliably ahead of PureSVD** (+0.0057, [+0.0025, +0.0088]) but **not distinguishable from item-item** (+0.0039, [-0.0001, +0.0080]).

## 4. What goes wrong (1): one validation window is one draw

Popularity can also be *recent* popularity: count only the last 30, 60, 90 or 180 days before the cutoff, on the idea that what sells now is what will sell next. Score every window on the validation window and on the test window:

In [5]:
windows = {"all history": None, "last 180 days": 180, "last 90 days": 90, "last 60 days": 60, "last 30 days": 30}
def popularity_ndcg(t, days):
    pop = rs.popularity(t.B) if days is None else recent_popularity(t, days)
    return discovery_ndcg(rs.broadcast(pop, n_customers), t)

flip = pd.DataFrame({"validation (Jun to Aug)": {w: popularity_ndcg(validation, d) for w, d in windows.items()},
                     "test (Sep to Nov)": {w: popularity_ndcg(test, d) for w, d in windows.items()}})
display(flip)
winner_on_validation = flip["validation (Jun to Aug)"].idxmax()
best_collaborative_on_validation = max(knn_grid.values())
print(f"best window on validation: {winner_on_validation} ({flip['validation (Jun to Aug)'].max():.4f}), "
      f"better than the best item-item model on validation ({best_collaborative_on_validation:.4f})")

,validation (Jun to Aug),test (Sep to Nov)
all history,0.0429,0.0436
last 180 days,0.0528,0.0467
last 90 days,0.0550,0.0511
last 60 days,0.0790,0.0519
last 30 days,0.1010,0.0531


best window on validation: last 30 days (0.1010), better than the best item-item model on validation (0.0872)


On the validation window the shorter the window the better, up to a factor of 2.4 (0.043 for all history, **0.101 for the last 30 days**), and 30-day popularity even beats the best item-item model on validation (0.0872). Anyone choosing a model on that table would conclude that recent popularity beats the collaborative models. On the test window the effect is much smaller (0.0436 to 0.0531, +22%) and the conclusion reverses: item-item scores 0.0820 there.

How typical is that validation window? The data allows nine cutoffs, from March 2010 to September 2011. Score all-history popularity, last-30-days popularity and item-item at each one:

In [6]:
cutoffs = ["2010-03-01", "2010-06-01", "2010-09-01", "2010-10-01", "2011-03-01", "2011-06-01", "2011-07-01", "2011-08-01", "2011-09-01"]
rows = {}
for cut in cutoffs:
    t = rs.make_purchase_task(sales, cut, days=90, index=index)
    rows[cut] = {"all-history popularity": discovery_ndcg(rs.broadcast(rs.popularity(t.B), n_customers), t),
                 "last 30 days": discovery_ndcg(rs.broadcast(recent_popularity(t, 30), n_customers), t),
                 "item-item": discovery_ndcg(rs.item_knn_scores(t.B, rs.item_similarity(t.B, k=best_k, shrink=best_lam)), t)}
by_cutoff = pd.DataFrame(rows).T.rename_axis("cutoff")
by_cutoff["best of the three"] = by_cutoff.idxmax(axis=1)
gain = by_cutoff["last 30 days"] - by_cutoff["all-history popularity"]
print(f"last 30 days beats all-history popularity at {(gain > 0).sum()} of {len(gain)} cutoffs; "
      f"the gain ranges from {gain.min():+.4f} to {gain.max():+.4f}")
print(f"item-item is the best of the three at {(by_cutoff['best of the three'] == 'item-item').sum()} of {len(by_cutoff)} cutoffs")
assert (gain > 0).all() and (by_cutoff["best of the three"] == "item-item").sum() >= 8 and by_cutoff.loc["2011-06-01", "best of the three"] == "last 30 days"
by_cutoff

last 30 days beats all-history popularity at 9 of 9 cutoffs; the gain ranges from +0.0022 to +0.0582
item-item is the best of the three at 8 of 9 cutoffs


,all-history popularity,last 30 days,item-item,best of the three
cutoff,,,,
2010-03-01,0.0656,0.0863,0.1632,item-item
2010-06-01,0.0713,0.0736,0.1481,item-item
2010-09-01,0.0576,0.0621,0.1147,item-item
2010-10-01,0.0502,0.0671,0.1038,item-item
2011-03-01,0.0487,0.0596,0.0884,item-item
2011-06-01,0.0429,0.1010,0.0872,last 30 days
2011-07-01,0.0413,0.0837,0.0855,item-item
2011-08-01,0.0406,0.0652,0.0842,item-item
2011-09-01,0.0436,0.0531,0.0820,item-item


The **direction** is steady and the **size** is not. Last-30-days popularity beats all-history popularity at all nine cutoffs, by anything from +0.002 to +0.058, so recency does help. But item-item is the best of the three at **eight of nine** cutoffs, and the single exception is **1 June 2011, the cutoff we happened to use for validation**. One validation window is one draw from a process whose effects vary more than twenty-fold, and ours happened to be the outlier. (Cutoffs a month apart have overlapping 90-day windows, so they are not independent evidence, but they are enough to see the pattern.) The remedy is the one from the time-series pages: choose among candidates over several rolling cutoffs and prefer the option that holds across them.

## 5. Repurchase on the test window

Now the other question: nothing is hidden, and a hit is any product bought in the window. The strongest simple baseline here is not popularity but the customer's **own history**: recommend what they bought most often. Compare it with the collaborative models, and also with a *careless* version of "own history" that only records whether they bought a product at all (0/1):

In [7]:
def repurchase_ndcg(scores):
    return existing_only(rs.rank_metrics(rs.top_k(scores, None, k=10), test.repeat)["ndcg"], test)     # nothing is hidden

own_frequency = np.asarray(test.counts.todense(), dtype=np.float32)
own_binary = np.asarray(test.B.todense(), dtype=np.float32)
repurchase = {
    "popularity (all history)": repurchase_ndcg(rs.broadcast(rs.popularity(test.B), n_customers)),
    "own history, 0/1": repurchase_ndcg(own_binary),
    "item-item": repurchase_ndcg(discovery["item-item"]),
    "implicit ALS": repurchase_ndcg(discovery["implicit ALS"]),
    "own history, by purchase count": repurchase_ndcg(own_frequency),
}
reference_r = repurchase["own history, by purchase count"]
rows = {}
for name, s in repurchase.items():
    d = rs.paired_difference(s, reference_r)
    rows[name] = {"test NDCG@10 (repurchase)": s.mean(),
                  "vs own history, by count": "" if s is reference_r else f"{d['diff']:+.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}]"}
repurchase_table = pd.DataFrame(rows).T
assert max(repurchase, key=lambda n: repurchase[n].mean()) == "own history, by purchase count"
assert all(rs.paired_difference(s_, reference_r)["hi"] < 0 for s_ in repurchase.values() if s_ is not reference_r)
repurchase_table

,test NDCG@10 (repurchase),"vs own history, by count"
popularity (all history),0.1139,"-0.2109 [-0.2222, -0.1996]"
"own history, 0/1",0.1369,"-0.1878 [-0.1979, -0.1777]"
item-item,0.2158,"-0.1090 [-0.1192, -0.0988]"
implicit ALS,0.2908,"-0.0340 [-0.0428, -0.0252]"
"own history, by purchase count",0.3248,


For existing customers, **no collaborative model beats the customer's own history ranked by purchase count** (0.3248). Implicit ALS, which was trained on the raw counts and so learns repeat behaviour, comes closest (0.2908, -0.034 with interval [-0.043, -0.025]); item-item reaches 0.2158 and popularity 0.1139. The careless version of own history, **0/1, scores 0.1369, less than half of the counted version**, with exactly the same information: every product ever bought ties, and the ties are broken by product order. For a pre-filled reorder form, the baseline to beat is a sort by purchase count.

## 6. Does the list make sense?

NDCG is a number; a person has to read the list. Take a customer with a real history who bought something new in the test window, and show what the item-item and ALS models would put in front of them, next to what they actually went on to buy.

In [8]:
names = sales.groupby("stock_code")["description"].agg(lambda s: s.mode().iat[0]).str.title()
history_size = np.asarray(test.B.sum(axis=1)).ravel()
new_count = np.asarray(test.new.sum(axis=1)).ravel()
candidates = np.flatnonzero((history_size >= 30) & (history_size <= 60) & (new_count >= 5))
u = int(candidates[len(candidates) // 2])                    # a typical such customer: the middle one

def listing(scores, n=5):
    top = rs.top_k(scores[u:u + 1], test.B[u:u + 1], k=n)[0]
    return pd.DataFrame({"product": [names[index.items[j]][:38] for j in top],
                         "bought in the next 90 days?": ["yes" if test.new[u, j] else "no" for j in top]})

usual = np.argsort(-own_frequency[u])[:4]
print(f"customer {index.users[u]:.0f}: {int(history_size[u])} different products so far, {int(new_count[u])} new ones in the window")
print("usually buys:", " | ".join(names[index.items[j]][:30] for j in usual))
display(listing(discovery["item-item"]).rename(columns={"product": "item-item recommends"}))
display(listing(discovery["implicit ALS"]).rename(columns={"product": "ALS recommends"}))

customer 15214: 46 different products so far, 51 new ones in the window
usually buys: Assorted Easter Gift Tags | Hot Baths Metal Sign | Wrap Wedding Day | Vintage Union Jack Apron


,item-item recommends,bought in the next 90 days?
0,Gin + Tonic Diet Metal Sign,no
1,Hand Over The Chocolate Sign,no
2,Please One Person Metal Sign,no
3,Pottering In The Shed Metal Sign,yes
4,I'M On Holiday Metal Sign,no


,ALS recommends,bought in the next 90 days?
0,Roses Regency Teacup And Saucer,no
1,Green Regency Teacup And Saucer,no
2,Please One Person Metal Sign,no
3,Red Hanging Heart T-Light Holder,no
4,Gin + Tonic Diet Metal Sign,no


Customer 15214 usually buys gift tags, wall signs and aprons. Item-item recommends five metal wall signs and one of them was bought in the window; the ALS model recommends teacups, a T-light holder and two signs, none bought. One customer proves nothing (the table above is the evidence, and it favours ALS), but reading a list is still worth doing: these look like things *this customer* might buy, and five of the ten slots in the two lists are the same product family, a preview of the diversity question on [page 8](08-beyond-accuracy.ipynb).

## 7. What goes wrong (2): who cannot be served

Two features of the *customers* limit any recommender here:

In [9]:
lines = np.asarray(test.counts.sum(axis=1)).ravel()
share_top5 = np.sort(lines)[::-1][: int(0.05 * (lines > 0).sum())].sum() / lines.sum()
buyers = np.asarray(test.repeat.sum(axis=1)).ravel() > 0
no_history = np.asarray(test.B.sum(axis=1)).ravel() == 0
print(f"the busiest 5% of customers account for {share_top5:.0%} of all purchase lines before the cutoff")
print(f"{(buyers & no_history).sum()} of the {buyers.sum():,} customers who buy in the test window ({(buyers & no_history).sum() / buyers.sum():.0%}) have no history at all")

everyone = rs.evaluate(rs.broadcast(rs.popularity(test.B), n_customers), test.B, test.new)["ndcg"]
newcomers = everyone[no_history[everyone.index.to_numpy()]]
print(f"popularity serves those {len(newcomers)} newcomers at NDCG@10 {newcomers.mean():.4f}; a collaborative model scores every product 0 for them")
assert newcomers.mean() > 1.5 * per_customer["popularity (all history)"].mean()

the busiest 5% of customers account for 37% of all purchase lines before the cutoff
598 of the 2,831 customers who buy in the test window (21%) have no history at all


popularity serves those 598 newcomers at NDCG@10 0.0997; a collaborative model scores every product 0 for them


Among the customers who do have a history, the benefit is not equal either. Split the scored customers into thirds by the number of different products they have bought so far:

In [10]:
history_size = np.asarray(test.B.sum(axis=1)).ravel()
scored = per_customer["implicit ALS"].index.to_numpy()
third = pd.qcut(pd.Series(history_size[scored]).rank(method="first"), 3, labels=["shortest history", "middle", "longest history"]).to_numpy()
rows = {}
for g in ("shortest history", "middle", "longest history"):
    members = scored[third == g]
    gain = rs.paired_difference(per_customer["implicit ALS"][members], per_customer["popularity (all history)"][members])
    rows[g] = {"customers": len(members), "products bought so far (median)": int(np.median(history_size[members])),
               "popularity": per_customer["popularity (all history)"][members].mean(), "item-item": per_customer["item-item"][members].mean(),
               "implicit ALS": per_customer["implicit ALS"][members].mean(),
               "ALS minus popularity": f"{gain['diff']:+.4f} [{gain['lo']:+.4f}, {gain['hi']:+.4f}]"}
    assert gain["lo"] > 0                                                                          # guards the reading below
pd.DataFrame(rows).T

,customers,products bought so far (median),popularity,item-item,implicit ALS,ALS minus popularity
shortest history,714,23,0.0424,0.0776,0.0839,"+0.0415 [+0.0314, +0.0516]"
middle,714,74,0.0446,0.0810,0.0854,"+0.0408 [+0.0312, +0.0504]"
longest history,714,190,0.0440,0.0875,0.0886,"+0.0446 [+0.0359, +0.0533]"


The busiest 5% of customers account for **37% of all purchase lines**, so the heaviest accounts shape what the model learns. And **21% of the customers who buy have no history**: a collaborative model scores every product 0 for them. A fallback serves them, and popularity does it well, at NDCG 0.0997, **more than twice what it scores for existing customers** (0.0436): newcomers buy the bestsellers, while established customers buy things specific to them.

Among existing customers the gain from personalising is **the same in all three thirds**: ALS beats popularity by +0.0415 for the shortest histories (median 23 products), +0.0408 for the middle third (74) and +0.0446 for the longest (190), each interval clearly above zero. Once a customer has any real history, a longer one adds little here. (A caution about populations: if the 598 customers with no history are left in the comparison, the group with fewer than 20 products shows popularity *beating* ALS, 0.080 against 0.028, only because the collaborative models output all zeros for them. Who is in the denominator decides the answer; say who you scored.)

<div class="admonition warning">
<p class="admonition-title">Watch out</p>
<p>Before building a collaborative model, ask what the strongest <em>non-collaborative</em> baseline for the same question is. For repurchase it is the customer's own history; for discovery it is popularity. A model that cannot beat the right baseline is not adding value, however good its NDCG looks next to a weak one.</p>
</div>

## Decision guide

| Question | Start from | Then try | Why |
|---|---|---|---|
| What will they reorder? | Own history by purchase count | Implicit ALS trained on counts | Customers reorder what they already buy |
| What could we introduce? | Popularity | Item-item or implicit ALS | Collaborative signals add information only for items the customer lacks |
| Seasonal assortment | Check several validation windows | A recent-popularity window chosen over several cutoffs | One validation window can mislead |
| Customer with no history | Popularity, or a segment-level list | Cold-start methods ([page 7](07-cold-start.ipynb)) | Collaborative models need history |

## Check yourself

<details class="question">
<summary>1. Why can "customer did not buy product X" not be treated as "customer dislikes X"?</summary>

The customer may never have seen X, may not need it yet, or may buy it from another supplier. The missing cell mixes "no interest" and "no opportunity". Implicit-feedback ALS handles this by treating missing cells as weak evidence (confidence 1) and observed cells as strong (confidence $1 + \alpha x$), never as a certain negative.

</details>

<details class="question">
<summary>2. Own history scored 0/1 does much worse than own history scored by purchase count. Both contain the same products. Why?</summary>

With 0/1 scores every product the customer ever bought ties at the same score, and the ties are broken by product order, which has nothing to do with the customer. Purchase counts put the products they buy most at the top. The lesson is general: a score with many ties cannot be evaluated until the tie-break is a meaningful one.

</details>

## Exercises

<details class="question">
<summary>Exercise 1 (easy): how many of the existing customers scored in the discovery table are among the busiest 5% by purchase lines?</summary>

```python
scored = per_customer["implicit ALS"].index                         # customer rows that were scored
cutoff = np.sort(lines)[::-1][int(0.05 * (lines > 0).sum())]
print((lines[scored] >= cutoff).sum(), "of", len(scored))
```

The answer is 229 of the 2,142 scored customers (11%), about twice their 5% share of customers: heavy customers are more likely to buy something new in a 90-day window, so they are over-represented among the scored.

</details>

<details class="question">
<summary>Exercise 2 (medium): does the ranking of models in the discovery table change if you score only existing customers with fewer than 20 different products in their history?</summary>

```python
light = [c for c in per_customer["implicit ALS"].index if history_size[c] < 20]
for name, s in per_customer.items():
    print(name, s.loc[light].mean())
```

There are 288 such customers: popularity scores 0.0385, item-item 0.0796 and ALS 0.0850, and ALS beats popularity by +0.0465 with an interval of [+0.0298, +0.0632]. The ranking does not change. Note the number of customers behind each mean, and see the caution at the end of section 7 about what happens if the newcomers are left in.

</details>

<details class="question">
<summary>Exercise 3 (hard): add the cutoffs 1 April 2011 and 1 May 2011 to the table in section 4. Does last-30-days popularity ever beat item-item again?</summary>

```python
for cut in ["2011-04-01", "2011-05-01"]:
    t = rs.make_purchase_task(sales, cut, days=90, index=index)
    print(cut, discovery_ndcg(rs.broadcast(recent_popularity(t, 30), n_customers), t),
          discovery_ndcg(rs.item_knn_scores(t.B, rs.item_similarity(t.B, k=best_k, shrink=best_lam)), t))
```

No. Item-item wins at both (0.0849 against 0.0689 on 1 April, 0.0874 against 0.0596 on 1 May). Of the eleven cutoffs we have now looked at, the validation cutoff of 1 June 2011 is the only one where recent popularity is ahead. Decide in advance how many cutoffs you need to agree before you trust a choice.

</details>

## Recap

- **Implicit feedback** has no negatives: a purchase is a positive signal of unknown strength, a missing cell is "unknown".
- Split by **date**, and score **repurchase** and **discovery** separately: they are different questions with different strongest baselines.
- For repurchase the baseline to beat is **the customer's own history by purchase count**; for discovery it is popularity.
- How to weight purchases (0/1, log, raw) mattered little here; the baselines and the evaluation framing mattered a lot.
- **One validation window is one draw:** of nine cutoffs, the one used for validation was the only one where recent popularity beat item-item. Choose over several rolling cutoffs.
- One customer in five who buys in the window has no history; a popularity fallback serves them, and **who is in the denominator** (existing customers only, or everyone) decides what a comparison says. A few heavy accounts dominate the data.